In [0]:
%sql
USE CATALOG vertem;
USE SCHEMA workshop_dataprep;

In [0]:
SELECT * FROM cdc_clientes_raw ORDER BY ts, cliente_id;

In [0]:
SELECT * FROM dim_cliente_scd1 ORDER BY cliente_id;

In [0]:
SELECT cliente_id, nome, cidade, segmento, __START_AT, __END_AT
FROM dim_cliente_scd2
ORDER BY cliente_id, __START_AT;

In [0]:
SELECT 'SCD 1' AS tipo, cliente_id, segmento,
       CAST(NULL AS TIMESTAMP) AS __START_AT,
       CAST(NULL AS TIMESTAMP) AS __END_AT
FROM dim_cliente_scd1
WHERE cliente_id = 1
UNION ALL
SELECT 'SCD 2' AS tipo, cliente_id, segmento, __START_AT, __END_AT
FROM dim_cliente_scd2
WHERE cliente_id = 1
ORDER BY tipo, __START_AT;

In [0]:
SELECT 'dim_cliente_scd1' AS tabela, COUNT(*) AS linhas
FROM dim_cliente_scd1
UNION ALL
SELECT 'dim_cliente_scd2', COUNT(*)
FROM dim_cliente_scd2
UNION ALL
SELECT 'dim_cliente_scd2 (vigentes)', COUNT(*)
FROM dim_cliente_scd2 WHERE __END_AT IS NULL;

In [0]:
WITH vendas AS (
  SELECT 1 AS venda_id, 1 AS cliente_id, TIMESTAMP '2026-01-20 00:00:00' AS data_venda, 1000.00 AS valor
  UNION ALL
  SELECT 2, 1, TIMESTAMP '2026-02-25 00:00:00', 1500.00
  UNION ALL
  SELECT 3, 1, TIMESTAMP '2026-04-01 00:00:00', 2000.00
)
SELECT v.venda_id, v.data_venda, v.valor,
       d.nome,
       d.segmento AS segmento_na_data_da_venda
FROM vendas v
JOIN dim_cliente_scd2 d
  ON v.cliente_id = d.cliente_id
 AND v.data_venda >= d.__START_AT
 AND (v.data_venda < d.__END_AT OR d.__END_AT IS NULL)
ORDER BY v.venda_id;